# Approach 4 v2 — Hybrid Linear + Crop-Conditional Prototype (fixed)

**What this notebook fixes vs. the previous version:**
1. `pair_scores` returns `[B, N]` for crop, disease and linear branches — no `.diag()`, no `[8,D]×[323,D]` broadcast error.
2. **Shared trainable parameters.** A trainable projection sits on top of frozen DINOv2. AM-Softmax updates `proj + heads`, and prototypes are rebuilt from `proj` outputs — so training actually changes what inference uses. No more isolated `crop_am`/`dis_am`.
3. **Two-stage official evaluation.** Stage A generates predictions from images only (labels untouched). Stage B reads `pair_row` afterwards. Candidate pool is fixed at 333 (323 seen + 10 unseen identities).
4. **Calibration separated from hybrid mixing.** `lam` = linear/prototype blend (selected on seen_pairs validation). `gamma` = candidate-prior calibration coefficient (selected on train-only proxy). Main table reports raw (`gamma=0`), calibrated table reported separately.
5. **Header clarity.** All images are ranked against 333 candidates. Metrics keys are `seen_images_top1_on_333`, `unseen_images_top1_on_333`, `all_images_top1_on_333`.

**Why each function is added (report justification):**
- **Prototype geometry** — on frozen backbones, prototype heads generalize better than linear probes on unseen class combinations (systematic study, 118 configs: avg +12.7 pts over linear; DINOv2+proto reaches 33.80% in 1-shot). The pair score becomes `cos(f, crop_proto) + cos(f, disease_proto)`, which does not require joint supervision on seen pairs.
- **AM-Softmax on the projection** — compressed intra-class features improve prototype quality. Open-set plant disease literature reports +15.21 AUROC/OSCR with AM-Softmax.
- **Crop-conditional disease prototypes** — hard routing in AdapterMoE shows per-crop disease isolation reduces cross-crop confusion. Here it's a fallback: use `crop_disease_proto[c][d]` when observed in training, otherwise fall back to `global_disease_proto[d]`.
- **Fixed-λ fusion** — the same systematic study found learned routing gives no additional gain on frozen features; fixed 0.5/0.5 matches or beats adaptive routers.
- **Calibration (gamma)** — prior correction corrects candidate-frequency imbalance introduced by 323 vs 10 pool. `gamma=0` is main; `gamma>0` on train-only proxy is reported separately to show the calibration did not change model selection.

**Instructor protocol.** Final evaluation on 333 candidates: 323 seen + 10 unseen. Unseen pair identities are read at prediction time (allowed), unseen per-image labels accessed only in Stage B for metrics.

In [ ]:
# ---- 1. Setup ----
import copy, gc, json, sys
from pathlib import Path
from datetime import datetime

import timm
import torch
import torch.nn.functional as F
from torch import nn

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'common' / 'dataset.py').is_file())
sys.path.insert(0, str(ROOT))

from common import config as cfg
from common.dataset import get_loaders, get_pairs, get_final_test_loaders

MODEL = 'vit_base_patch14_dinov2.lvd142m'
MODEL_NAME = 'approach4_hybrid_proto_crop_cond_v2'
DEBUG = True

HP = dict(
    backbone=MODEL,
    batch_size=8 if DEBUG else 16,
    epochs=1 if DEBUG else 12,
    proj_dim=256,
    lr=1e-3,
    weight_decay=1e-4,
    am_margin=0.35,
    am_scale=30.0,
    hybrid_lambda=0.5,
    norm='imagenet',
    seed=cfg.SEED,
)

cfg.NUM_WORKERS = 0
cfg.set_seed(cfg.SEED)
RUN = MODEL_NAME + ('_debug' if DEBUG else '') + '_' + datetime.now().strftime('%Y%m%d_%H%M%S')
OUT = cfg.CHECKPOINT_DIR / RUN
device = cfg.DEVICE
print('Device:', device, '| Model:', MODEL_NAME, '| Debug:', DEBUG)

## 2. Data (train CSV only)

Only `train_loader` and `val_loader` are touched here. `seen`/`unseen` official loaders are not created in this cell — they are deferred to Stage A of final evaluation.

In [ ]:
def make_loaders():
    cfg.set_seed(cfg.SEED)
    tr, va, _, _ = get_loaders(HP['batch_size'], norm=HP['norm'], debug=DEBUG)
    tr.pin_memory = va.pin_memory = False
    assert not set(tr.dataset.frame.image) & set(va.dataset.frame.image)
    return tr, va

train_loader, val_loader = make_loaders()
seen_pairs = get_pairs()  # (323, 2)
print(f'Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Seen pairs: {seen_pairs.shape}')

## 3. Model: frozen DINOv2 + trainable projection + cosine heads + prototype bank

**Why a projection sits between backbone and heads?** The AM-Softmax training signal must reach the features that prototypes are computed from. With a frozen backbone, that means a trainable projection. Prototypes are `mean(proj(backbone(x)))` per class, so as `proj` and the heads are trained, prototypes evolve and match the heads' feature geometry.

**Prototype bank**
- `crop_proto[c]` — mean projected feature per crop
- `global_disease_proto[d]` — mean projected feature per disease
- `crop_disease_proto[c,d]` — mean projected feature per (crop, disease) when observed in training
- `crop_disease_mask[c,d]` — whether `(c,d)` was observed

For a candidate pair `(c,d)`, if `(c,d)` was observed, disease prototype is `crop_disease_proto[c,d]`. Otherwise we fall back to `global_disease_proto[d]` — this is exactly what makes unseen combinations scoreable.

In [ ]:
class CosineHead(nn.Module):
    """Cosine classifier shared between training (AM-Softmax) and inference (log-softmax)."""
    def __init__(self, proj_dim, n_classes):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_classes, proj_dim))
        nn.init.xavier_uniform_(self.weight)

    def cosine_logits(self, feats):
        return F.normalize(feats, dim=-1) @ F.normalize(self.weight, dim=-1).T  # [B, C]

    def am_loss(self, feats, labels, margin, scale):
        cos = self.cosine_logits(feats)
        one_hot = F.one_hot(labels, num_classes=self.weight.size(0)).float()
        target = cos - margin * one_hot
        return F.cross_entropy(scale * target, labels)


class HybridModel(nn.Module):
    def __init__(self, proj_dim):
        super().__init__()
        self.backbone = timm.create_model(MODEL, pretrained=True, num_classes=0, img_size=224)
        self.backbone.requires_grad_(False)
        feat_dim = self.backbone.num_features
        self.proj = nn.Sequential(
            nn.Linear(feat_dim, proj_dim),
            nn.GELU(),
            nn.LayerNorm(proj_dim),
        )
        self.crop_head = CosineHead(proj_dim, cfg.NUM_CROPS)
        self.disease_head = CosineHead(proj_dim, cfg.NUM_DISEASES)
        self.register_buffer('crop_proto', torch.zeros(cfg.NUM_CROPS, proj_dim))
        self.register_buffer('global_disease_proto', torch.zeros(cfg.NUM_DISEASES, proj_dim))
        self.register_buffer('crop_disease_proto', torch.zeros(cfg.NUM_CROPS, cfg.NUM_DISEASES, proj_dim))
        self.register_buffer('crop_disease_mask', torch.zeros(cfg.NUM_CROPS, cfg.NUM_DISEASES, dtype=torch.bool))

    def forward(self, images):
        with torch.no_grad():
            raw = self.backbone(images).float()
        return self.proj(raw)


@torch.no_grad()
def build_prototypes(model, loader):
    """Rebuild prototypes from projected features (the ones training actually updates)."""
    model.eval()
    D = model.crop_proto.shape[1]
    crop_sum = torch.zeros(cfg.NUM_CROPS, D, device=device)
    crop_cnt = torch.zeros(cfg.NUM_CROPS, device=device)
    dis_sum = torch.zeros(cfg.NUM_DISEASES, D, device=device)
    dis_cnt = torch.zeros(cfg.NUM_DISEASES, device=device)
    cd_sum = torch.zeros(cfg.NUM_CROPS, cfg.NUM_DISEASES, D, device=device)
    cd_cnt = torch.zeros(cfg.NUM_CROPS, cfg.NUM_DISEASES, device=device)

    for images, crop_lbl, dis_lbl, _ in loader:
        feats = model(images.to(device))
        crop_lbl = crop_lbl.to(device); dis_lbl = dis_lbl.to(device)
        crop_sum.index_add_(0, crop_lbl, feats)
        crop_cnt.index_add_(0, crop_lbl, torch.ones_like(crop_lbl, dtype=torch.float))
        dis_sum.index_add_(0, dis_lbl, feats)
        dis_cnt.index_add_(0, dis_lbl, torch.ones_like(dis_lbl, dtype=torch.float))
        for c, d, f in zip(crop_lbl.tolist(), dis_lbl.tolist(), feats):
            cd_sum[c, d] += f
            cd_cnt[c, d] += 1

    model.crop_proto.copy_(crop_sum / crop_cnt.clamp_min(1).unsqueeze(1))
    model.global_disease_proto.copy_(dis_sum / dis_cnt.clamp_min(1).unsqueeze(1))
    model.crop_disease_proto.copy_(cd_sum / cd_cnt.clamp_min(1).unsqueeze(-1))
    model.crop_disease_mask.copy_(cd_cnt > 0)
    model.crop_proto.copy_(F.normalize(model.crop_proto, dim=1))
    model.global_disease_proto.copy_(F.normalize(model.global_disease_proto, dim=1))
    normed = F.normalize(model.crop_disease_proto, dim=-1)
    model.crop_disease_proto.copy_(
        torch.where(model.crop_disease_mask.unsqueeze(-1), normed, model.crop_disease_proto))
    return int(model.crop_disease_mask.sum().item())


@torch.no_grad()
def pair_scores(model, feats, pairs, lam, gamma=0.0, prior=None):
    """
    feats:  [B, D] projected features
    pairs:  [N, 2] candidate (crop_id, disease_id)
    return: [B, N] scores — every branch preserves batch x candidates
    """
    feats_n = F.normalize(feats, dim=-1)
    crop_ids, dis_ids = pairs.unbind(1)             # [N], [N]

    # --- Linear branch (same head weights used during training) ---
    cos_crop = model.crop_head.cosine_logits(feats)   # [B, C]
    cos_dis = model.disease_head.cosine_logits(feats) # [B, K]
    logp_crop = F.log_softmax(cos_crop * HP['am_scale'], dim=-1)   # [B, C]
    logp_dis = F.log_softmax(cos_dis * HP['am_scale'], dim=-1)     # [B, K]
    linear_score = logp_crop[:, crop_ids] + logp_dis[:, dis_ids]   # [B, N]

    # --- Prototype branch ---
    sim_crop = feats_n @ model.crop_proto[crop_ids].T              # [B, N]
    cd_proto = model.crop_disease_proto[crop_ids, dis_ids]         # [N, D]
    g_proto = model.global_disease_proto[dis_ids]                  # [N, D]
    has_cd = model.crop_disease_mask[crop_ids, dis_ids]            # [N]
    dis_proto = torch.where(has_cd.unsqueeze(1), cd_proto, g_proto)  # [N, D]
    sim_dis = feats_n @ dis_proto.T                                # [B, N]
    proto_score = sim_crop + sim_dis                               # [B, N]

    # --- Hybrid + prior calibration (gamma=0 => raw) ---
    score = lam * proto_score + (1 - lam) * linear_score           # [B, N]
    if gamma > 0 and prior is not None:
        score = score + gamma * prior.unsqueeze(0)                 # prior: [N]
    return score

## 4. Training

Only `proj`, `crop_head.weight`, `disease_head.weight` are trainable. Backbone stays frozen. Prototypes are rebuilt at the start of each epoch (before training) and once after — the second rebuild reflects the newly updated `proj`, so validation sees prototypes consistent with the current heads.

In [ ]:
def make_model():
    cfg.set_seed(cfg.SEED)
    return HybridModel(HP['proj_dim']).to(device)


def train_one_epoch(model, optimizer, loader):
    model.train()
    model.backbone.eval()
    total, count = 0.0, 0
    for images, crop_lbl, dis_lbl, _ in loader:
        images = images.to(device); crop_lbl = crop_lbl.to(device); dis_lbl = dis_lbl.to(device)
        feats = model(images)                                      # grads flow: proj + heads
        loss = model.crop_head.am_loss(feats, crop_lbl, HP['am_margin'], HP['am_scale'])
        loss = loss + model.disease_head.am_loss(feats, dis_lbl, HP['am_margin'], HP['am_scale'])
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(images); count += len(images)
    return total / max(count, 1)


@torch.no_grad()
def val_pair_top1(model, loader, pairs, lam, gamma=0.0, prior=None):
    model.eval()
    correct, n = 0, 0
    for images, _, _, pair_lbl in loader:
        feats = model(images.to(device))
        scores = pair_scores(model, feats, pairs.to(device), lam, gamma, prior)
        correct += int((scores.argmax(1) == pair_lbl.to(device)).sum().item())
        n += len(pair_lbl)
    return 100.0 * correct / max(n, 1)


OUT.mkdir(parents=True, exist_ok=False)
model = make_model()
optimizer = torch.optim.AdamW(
    list(model.proj.parameters()) + list(model.crop_head.parameters()) + list(model.disease_head.parameters()),
    lr=HP['lr'], weight_decay=HP['weight_decay'])

best_val, best_state, history = -1.0, None, []
for epoch in range(1, HP['epochs'] + 1):
    n_cells = build_prototypes(model, train_loader)
    loss = train_one_epoch(model, optimizer, train_loader)
    n_cells = build_prototypes(model, train_loader)   # refresh after training step
    val_top1 = val_pair_top1(model, val_loader, seen_pairs, HP['hybrid_lambda'])
    rec = dict(epoch=epoch, loss=loss, val_top1=val_top1, populated_cells=n_cells)
    history.append(rec); print(rec)
    if val_top1 > best_val:
        best_val = val_top1
        best_state = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)
build_prototypes(model, train_loader)   # final refresh with best weights
torch.save(dict(model=MODEL_NAME, hp=HP, state_dict=best_state, best_val=best_val, history=history),
           OUT / (MODEL_NAME + '.pt'))
print(f'Best val top-1: {best_val:.2f}%')

## 5. Ablation on validation split (train CSV only)

Isolates each function's contribution. All cells evaluated against 323 seen candidates (this is a proxy, not official).

In [ ]:
ablation = {}
ablation['linear_only'] = val_pair_top1(model, val_loader, seen_pairs, lam=0.0)
orig_mask = model.crop_disease_mask.clone()
model.crop_disease_mask.zero_()
ablation['global_proto_only'] = val_pair_top1(model, val_loader, seen_pairs, lam=1.0)
model.crop_disease_mask.copy_(orig_mask)
ablation['crop_cond_proto_only'] = val_pair_top1(model, val_loader, seen_pairs, lam=1.0)
ablation['hybrid_0.5'] = val_pair_top1(model, val_loader, seen_pairs, lam=0.5)
for k, v in ablation.items():
    print(f'{k:25s} val top-1 (323 proxy): {v:.2f}%')

## 6. Freeze `lam` and `gamma` on train-only validation

`lam` sweeps the linear/prototype blend. `gamma` sweeps candidate-prior calibration. Both are selected on `val_loader` (derived from the training CSV), **never on the official test CSVs**, then saved to `frozen_settings.pt` so final evaluation cannot silently change them.

In [ ]:
# --- lam sweep on seen_pairs validation ---
lam_grid = [0.0, 0.25, 0.5, 0.75, 1.0]
lam_scores = {lam: val_pair_top1(model, val_loader, seen_pairs, lam=lam) for lam in lam_grid}
best_lam = max(lam_scores, key=lam_scores.get)
print('lam sweep:', {k: round(v, 3) for k, v in lam_scores.items()}, '| best:', best_lam)

# --- candidate prior from training frequency ---
train_frame = train_loader.dataset.frame
prior = torch.zeros(len(seen_pairs), device=device)
if 'pair_row' in train_frame:
    for idx, cnt in train_frame['pair_row'].value_counts().items():
        prior[int(idx)] = float(cnt)
prior = prior + 1.0
prior = prior / prior.sum()
prior = torch.log(prior)
prior = prior - prior.mean()

# --- gamma sweep ---
gamma_grid = [0.0, 0.5, 1.0, 2.0]
gamma_scores = {
    g: val_pair_top1(model, val_loader, seen_pairs, lam=best_lam, gamma=g, prior=prior)
    for g in gamma_grid
}
best_gamma = max(gamma_scores, key=gamma_scores.get)
print('gamma sweep:', {k: round(v, 3) for k, v in gamma_scores.items()}, '| best:', best_gamma)

FROZEN = dict(lam=float(best_lam), gamma=float(best_gamma), prior=prior.cpu())
torch.save(FROZEN, OUT / 'frozen_settings.pt')
print('Frozen:', dict(lam=best_lam, gamma=best_gamma))

## 7. Final evaluation — Stage A (predictions only) + Stage B (labels only)

**Stage A** generates predictions by iterating with `for images, *_ in loader` — only images are accessed. All 333 candidate identities are formed from `get_pairs(final_evaluation=True)` (allowed: pair identities, not per-image labels). Predictions are stored and **cannot change** afterwards.

**Stage B** reads `pair_row` from the same loaders' frames to compute metrics.

**Table headers are explicit that all images rank against 333 candidates**:
- `seen_images_top1_on_333`
- `unseen_images_top1_on_333`
- `all_images_top1_on_333`
- `harmonic_mean_seen_unseen_top1`

**Two score sets reported:** raw (`gamma=0`) as main table, calibrated (`gamma=best_gamma`) as a separate row, showing calibration was chosen on train-only proxy.

In [ ]:
RUN_FINAL = False

if RUN_FINAL:
    if DEBUG:
        raise ValueError('Final evaluation requires a non-DEBUG run.')

    # Reload the best checkpoint + frozen settings
    ckpt = torch.load(OUT / (MODEL_NAME + '.pt'), map_location='cpu', weights_only=False)
    FROZEN = torch.load(OUT / 'frozen_settings.pt', map_location='cpu', weights_only=False)
    lam = FROZEN['lam']
    gamma = FROZEN['gamma']
    prior_frozen = FROZEN['prior'].to(device)

    model = make_model()
    model.load_state_dict(ckpt['state_dict'])
    model.to(device).eval()
    build_prototypes(model, train_loader)

    # Candidate pool: 323 seen identities + 10 unseen identities (allowed)
    final_pairs = get_pairs(final_evaluation=True)
    assert final_pairs.shape[0] == 333, final_pairs.shape
    seen_loader, unseen_loader = get_final_test_loaders(HP['batch_size'], norm=HP['norm'], final_evaluation=True)

    # ---------------- Stage A: predictions only ----------------
    @torch.no_grad()
    def generate_scores(loader):
        out = []
        for images, *_ in loader:            # labels never read here
            feats = model(images.to(device))
            out.append(pair_scores(model, feats, final_pairs.to(device), lam, gamma=0.0).cpu())  # raw
        return torch.cat(out)

    seen_raw = generate_scores(seen_loader)         # [N_seen_images, 333]
    unseen_raw = generate_scores(unseen_loader)     # [N_unseen_images, 333]
    # Predictions are now frozen.

    # ---------------- Stage B: labels only ----------------
    seen_labels = torch.as_tensor(seen_loader.dataset.frame['pair_row'].to_numpy())
    unseen_labels = torch.as_tensor(unseen_loader.dataset.frame['pair_row'].to_numpy())

    def topk(scores, labels, k):
        return 100.0 * (scores.topk(k, dim=1).indices == labels.unsqueeze(1)).any(1).float().mean().item()

    # Raw (gamma=0) — main table
    raw = dict(
        seen_images_top1_on_333=topk(seen_raw, seen_labels, 1),
        seen_images_top5_on_333=topk(seen_raw, seen_labels, 5),
        unseen_images_top1_on_333=topk(unseen_raw, unseen_labels, 1),
        unseen_images_top5_on_333=topk(unseen_raw, unseen_labels, 5),
    )
    raw['all_images_top1_on_333'] = topk(torch.cat([seen_raw, unseen_raw]),
                                         torch.cat([seen_labels, unseen_labels]), 1)
    raw['all_images_top5_on_333'] = topk(torch.cat([seen_raw, unseen_raw]),
                                         torch.cat([seen_labels, unseen_labels]), 5)
    raw['harmonic_mean_seen_unseen_top1'] = (
        2 * raw['seen_images_top1_on_333'] * raw['unseen_images_top1_on_333']
        / (raw['seen_images_top1_on_333'] + raw['unseen_images_top1_on_333'] + 1e-12))

    # Calibrated (gamma > 0) — reported separately, settings frozen in step 6
    seen_cal = seen_raw + gamma * prior_frozen.unsqueeze(0).cpu()
    unseen_cal = unseen_raw + gamma * prior_frozen.unsqueeze(0).cpu()
    cal = dict(
        seen_images_top1_on_333=topk(seen_cal, seen_labels, 1),
        unseen_images_top1_on_333=topk(unseen_cal, unseen_labels, 1),
    )
    cal['harmonic_mean_seen_unseen_top1'] = (
        2 * cal['seen_images_top1_on_333'] * cal['unseen_images_top1_on_333']
        / (cal['seen_images_top1_on_333'] + cal['unseen_images_top1_on_333'] + 1e-12))

    report = dict(
        model=MODEL_NAME,
        protocol=dict(
            candidates_total=333, seen_candidates=323, unseen_candidates=10,
            seen_images=int(len(seen_labels)), unseen_images=int(len(unseen_labels)),
            metric_note='all images ranked against 333 candidate pairs',
        ),
        selected=dict(lam=lam, gamma=gamma),
        raw_gamma0=raw,
        calibrated=cal,
    )
    print(json.dumps(report, indent=2))

    # Seen confusion matrix over 333 candidates
    from sklearn.metrics import confusion_matrix
    import pandas as pd
    cm = confusion_matrix(seen_labels.numpy(), seen_raw.argmax(1).numpy(), labels=list(range(333)))
    pd.DataFrame(cm).to_csv(OUT / 'seen_confusion_matrix.csv', index=False)
    (OUT / 'final_results.json').write_text(json.dumps(report, indent=2))
    print('Saved to', OUT)
else:
    print('Final evaluation disabled. Train, ablate, freeze lam+gamma, then set RUN_FINAL=True.')

## 8. Report mapping (what to claim and how to justify)

| Function | Metric it should move | How to justify |
|---|---|---|
| Hybrid linear+prototype head | `unseen_images_top1_on_333`, `harmonic_mean` | Compare `linear_only` vs `hybrid_0.5` in ablation. Prototype geometry gives seen-combo-independent scoring. |
| AM-Softmax on shared projection | Prototype quality → unseen accuracy | `proj` is trainable; prototypes are computed from `proj`. Compare training loss curves + prototype sanity. |
| Crop-conditional disease prototypes | Unseen disease-side accuracy | `crop_cond_proto_only` vs `global_proto_only` in ablation. |
| Fixed-λ fusion | Stability vs 3C's pairwise fusion | Report harmonic mean and parameter count vs 3C. |
| Calibration gamma | Small boost with prior correction | Show `raw_gamma0` as main table; `calibrated` as supplementary. Justify it did not change model selection. |

**If any component does not beat its targeted metric, report honestly** — a negative ablation is a valid finding and strengthens the final model's justification.